# 30. 평가, 결합, 제출 검사

담당: 수정
목표: GPU 결과를 기다리지 않고 가짜 확률로 평가와 병합, 결합, 제출 검사 경로를 먼저 완성한 뒤,
같은 셀을 실제 예측 파일로 돌려서 최종 제출을 만드는 것입니다.

첫 30분 안에 앞부분을 끝내 두면, 준혁과 유림의 결과가 나오는 즉시 제출을 준비할 수 있습니다.

## 통과 기준(PASS)
1. 가짜 확률로 병합, 평가, 결합, 제출, 검사가 모두 종료 코드 0으로 끝납니다.
2. 일부러 망가뜨린 파일에서 검사기가 실패를 잡아냅니다.
3. 실제 예측 파일로 같은 경로를 돌려서 제출 후보를 만들었습니다.

## 넘겨줄 내용
제출 파일 경로와 해시, 실험 기록 표입니다.

### 명령 인자에 대한 안내

이 노트북의 명령은 `vqa/` 와 `scripts/` 에 실제로 구현된 인자에 맞춰 작성했습니다. 그래도 코어
코드가 계속 바뀌는 동안에는 인자 이름이 달라질 수 있습니다. 셀이 `unrecognized arguments` 로
실패하면 먼저 해당 명령에 `--help` 를 붙여서 실제 인자 이름을 확인하고, 셀 안의 문자열만 고치면
됩니다. 노트북의 다른 부분은 고치지 않아도 됩니다. 고친 내용은 팀 채널에 알려서 다른 사람이 같은
곳에서 막히지 않게 하십시오.

In [ ]:
# ===== 파라미터 =====
PROJECT_ROOT = r"."
DATA_ROOT = r"."
RUN_ROOT = r"./artifacts/runs"
LOCAL_SCRATCH = r"./artifacts/scratch"
MODEL_ROOT = r""

VAL_IDS_FILE = "fold_csv_files/val_fold0.csv"    # 팀 공식 검증 분할 (fold 0, 1,344문항)
FAKE_RUN_ID = "fake_smoke"
N_FAKE_SHARDS = 2
SEED = 7

# 실제 파일로 전환할 때 여기를 고칩니다. 아직 없는 출처는 빈 문자열로 둡니다.
REAL_VAL_BASE27 = "artifacts/runs/colab27b_base_bf16/preds/fold0_base.csv"
REAL_VAL_FT27 = "artifacts/runs/colab27b_ft_bf16/preds/fold0_ft.csv"
REAL_VAL_FT9 = "artifacts/runs/lab9b_ft_v1/preds/fold0_ft.csv"
REAL_TEST_BASE27 = "artifacts/runs/colab27b_base_bf16/preds/test_base.csv"
REAL_TEST_FT27 = "artifacts/runs/colab27b_ft_bf16/preds/test_ft.csv"
REAL_TEST_FT9 = "artifacts/runs/lab9b_ft_v1/preds/test_ft.csv"
TIE_EPSILON = 0.002   # 정확도 차이가 이 값 이하면 더 단순한 후보를 고릅니다.

EXPERIMENTS_LOG = "artifacts/experiments.jsonl"

In [ ]:
# --- 공통 부트스트랩: 경로, 환경 변수, 명령 실행 헬퍼 (수정할 일 거의 없음) ---
import os, sys, json, shlex, subprocess, time, csv, hashlib, random
from pathlib import Path

PROJECT_ROOT = Path(PROJECT_ROOT).expanduser().resolve()
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


def _setenv(name, value, create=False):
    if not value:
        return None
    p = Path(value).expanduser()
    if create:
        p.mkdir(parents=True, exist_ok=True)
    os.environ[name] = str(p)
    return p


DATA_ROOT_P = _setenv("VQA_DATA_ROOT", DATA_ROOT)
RUN_ROOT_P = _setenv("VQA_RUN_ROOT", RUN_ROOT, create=True)
SCRATCH_P = _setenv("VQA_LOCAL_SCRATCH", LOCAL_SCRATCH, create=True)
MODEL_ROOT_P = _setenv("VQA_MODEL_ROOT", MODEL_ROOT)

PY = sys.executable


def run(cmd, check=True, echo=True, quiet=False, hint=True):
    """명령을 실행하면서 출력을 그대로 보여 주고 종료 코드와 소요 시간을 돌려줍니다.

    quiet가 True이면 명령의 출력도, 뒤에 붙는 "[rc=...]" 줄도 찍지 않습니다. 상태를 조회하는
    호출은 실패가 정상적인 결과입니다. 예를 들어 `vqa.jobs status` 는 죽은 job에 대해 종료 코드
    1을 돌려줍니다. 그 줄을 그대로 찍으면 셀 출력 맨 앞에 "[rc=1]"이 남아서, 실제 실패로 읽히는
    오해를 부릅니다.
    """
    if isinstance(cmd, str):
        cmd = shlex.split(cmd)
    cmd = [str(x) for x in cmd]
    if echo:
        print("$ " + " ".join(shlex.quote(x) for x in cmd), flush=True)
    t0 = time.time()
    proc = subprocess.Popen(
        cmd, cwd=str(PROJECT_ROOT), stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, encoding="utf-8",
        errors="replace", bufsize=1,
    )
    buf = []
    for line in proc.stdout:
        buf.append(line)
        if not quiet:
            print(line, end="", flush=True)
    rc = proc.wait()
    dt = time.time() - t0
    if not quiet:
        print("[rc=%d] %.1f초" % (rc, dt), flush=True)
    if rc != 0:
        if hint:
            print("실패했습니다. 같은 명령에 --help 를 붙여서 인자 이름부터 확인하십시오.", flush=True)
        if check:
            raise RuntimeError("명령이 실패했습니다: " + " ".join(cmd))
    return {"rc": rc, "seconds": dt, "output": "".join(buf)}


def vqa(module, *args, **kw):
    """python -m vqa.<module> 을 실행합니다."""
    return run([PY, "-m", "vqa." + module] + [str(a) for a in args], **kw)


def script(name, *args, **kw):
    """scripts/<name> 을 실행합니다. name 에 / 가 있으면 저장소 루트 기준 경로로 봅니다."""
    path = Path(name) if "/" in name else Path("scripts") / name
    return run([PY, str(path)] + [str(a) for a in args], **kw)


def read_csv_rows(path):
    with open(path, newline="", encoding="utf-8") as fh:
        return list(csv.DictReader(fh))


def read_ids(path):
    return [ln.strip() for ln in Path(path).read_text(encoding="utf-8").splitlines() if ln.strip()]


def sha256_of(path, limit=None):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    digest = h.hexdigest()
    return digest[:limit] if limit else digest


print("PROJECT_ROOT      :", PROJECT_ROOT)
for _name in ("VQA_DATA_ROOT", "VQA_RUN_ROOT", "VQA_LOCAL_SCRATCH", "VQA_MODEL_ROOT"):
    print("%-18s: %s" % (_name, os.environ.get(_name, "(미설정)")))

## 1. 가짜 확률 만들기

`scripts/make_fake_predictions.py` 가 계약서에 정해진 예측 형식 그대로 만들어 줍니다. 컬럼은
`id, p_a, p_b, p_c, p_d, run_id` 이고, shard로 나누면 파일 이름이 `<이름>.shard{i}of{n}.csv` 가
됩니다.

`--mode oracle` 로 두면 `--gold` 의 정답을 `--accuracy` 비율만큼 맞히는 확률을 만듭니다. 평가기가
0이나 1이 아닌 값을 내는지 확인할 수 있기 때문에, 경로를 점검할 때는 이 방식이 편합니다.

In [ ]:
FAKE_DIR = RUN_ROOT_P / FAKE_RUN_ID / "preds"
FAKE_DIR.mkdir(parents=True, exist_ok=True)
GOLD = str(Path(os.environ["VQA_DATA_ROOT"]) / "train.csv")
SAMPLE = str(Path(os.environ["VQA_DATA_ROOT"]) / "sample_submission.csv")
TEST_CSV = str(Path(os.environ["VQA_DATA_ROOT"]) / "test.csv")

if not Path(VAL_IDS_FILE).exists():
    raise FileNotFoundError(
        VAL_IDS_FILE + " 이 없습니다. 저장소의 fold_csv_files/ 가 최신인지 확인하기 전에는 "
        "train.csv의 id 컬럼을 --ids로 넘겨서 형식만 확인하십시오."
    )

script("make_fake_predictions.py",
       "--ids", VAL_IDS_FILE,
       "--out", str(FAKE_DIR / "val_fake_base27.csv"),
       "--mode", "oracle", "--gold", GOLD, "--accuracy", "0.70",
       "--run-id", FAKE_RUN_ID + "_base27",
       "--shards", N_FAKE_SHARDS,
       "--seed", SEED)

shard_files = sorted(FAKE_DIR.glob("val_fake_base27.shard*of*.csv"))
print()
for f in shard_files:
    print(" 만들어진 shard:", f.name)

## 2. shard 병합

각 ID가 정확히 한 번씩만 들어 있어야 합니다. `--expect-ids` 로 기대 ID 목록을 넘기면 병합기가
빠진 ID와 남는 ID를 모두 잡아냅니다.

In [ ]:
MERGED = FAKE_DIR / "val_fake_base27.csv"
script("merge_shards.py",
       "--shards", *[str(f) for f in shard_files],
       "--expect-ids", VAL_IDS_FILE,
       "--out", str(MERGED))

## 3. 평가

정확도가 0.7 근처로 나오면 평가 경로가 정상입니다. 정확히 0이나 1이 나오면 ID 매칭이나 정답 컬럼
해석에 문제가 있는 것입니다.

가짜 확률로 하는 점검이므로 `--no-log` 를 붙여서 실험 기록에는 남기지 않습니다.

In [ ]:
Path("artifacts/eval").mkdir(parents=True, exist_ok=True)
script("evaluate_run.py",
       "--pred", str(MERGED),
       "--gold", GOLD,
       "--val-ids", VAL_IDS_FILE,
       "--json-out", "artifacts/eval/fake_val.json",
       "--wrong-out", "artifacts/eval/fake_val_wrong.csv",
       "--no-log")

## 4. 결합

`experiments/eval_tools/combine.py` 는 계획서 8절에서 미리 정한 네 후보만 비교합니다. 후보는 base27 단독,
ft27 단독, ft27 0.8에 base27 0.2를 더한 것, ft27 0.8에 ft9 0.2를 더한 것입니다. 가중치를 자유롭게
탐색하지 않는 이유는 validation에 과적합되는 것을 막기 위해서입니다.

각 확률 파일은 `--val-base27`, `--val-ft27`, `--val-ft9` 처럼 출처 이름을 붙여서 넘깁니다.
test 파일은 같은 이름에 `--test-` 를 붙입니다.

In [ ]:
for name, acc in (("ft27", 0.74), ("ft9", 0.66)):
    script("make_fake_predictions.py",
           "--ids", VAL_IDS_FILE,
           "--out", str(FAKE_DIR / ("val_fake_%s.csv" % name)),
           "--mode", "oracle", "--gold", GOLD, "--accuracy", str(acc),
           "--run-id", FAKE_RUN_ID + "_" + name,
           "--seed", SEED + 1,
           quiet=True)
    print("만들었습니다: val_fake_%s.csv (목표 정확도 %.2f)" % (name, acc))

for name in ("base27", "ft27", "ft9"):
    script("make_fake_predictions.py",
           "--ids", TEST_CSV,
           "--out", str(FAKE_DIR / ("test_fake_%s.csv" % name)),
           "--mode", "random",
           "--run-id", FAKE_RUN_ID + "_" + name,
           "--seed", SEED + 2,
           quiet=True)
    print("만들었습니다: test_fake_%s.csv" % name)

In [ ]:
FAKE_TEST_COMBINED = FAKE_DIR / "test_fake_combined.csv"
script("experiments/eval_tools/combine.py",
       "--val-base27", str(FAKE_DIR / "val_fake_base27.csv"),
       "--val-ft27", str(FAKE_DIR / "val_fake_ft27.csv"),
       "--val-ft9", str(FAKE_DIR / "val_fake_ft9.csv"),
       "--test-base27", str(FAKE_DIR / "test_fake_base27.csv"),
       "--test-ft27", str(FAKE_DIR / "test_fake_ft27.csv"),
       "--test-ft9", str(FAKE_DIR / "test_fake_ft9.csv"),
       "--gold", GOLD,
       "--val-ids", VAL_IDS_FILE,
       "--out", str(FAKE_TEST_COMBINED),
       "--val-out", str(FAKE_DIR / "val_fake_combined.csv"),
       "--report-out", "artifacts/eval/fake_combine_report.json",
       "--no-log")

## 5. 제출 파일과 검사기

결합한 test 확률을 제출 파일로 바꾸고 검사기를 돌립니다. ID 집합과 행 수, 순서, 값 범위를 모두
확인합니다. `vqa.submit` 은 파싱에 실패한 행을 임의로 `a` 로 채우지 않습니다.

In [ ]:
Path("artifacts/submission").mkdir(parents=True, exist_ok=True)
FAKE_SUB = "artifacts/submission/fake_smoke.csv"

vqa("submit", "--pred", str(FAKE_TEST_COMBINED),
    "--sample", SAMPLE, "--out", FAKE_SUB,
    "--margins-out", "artifacts/eval/fake_margins.csv")
script("check_submission.py", "--submission", FAKE_SUB, "--sample", SAMPLE)

## 6. 검사기가 실패를 잡는지 확인

일부러 망가뜨린 입력을 넣어서 검사가 실제로 막아 주는지 봅니다. 여기에서 전부 통과해 버리면
검사기를 신뢰할 수 없습니다.

두 방향으로 확인합니다. 먼저 예측 확률 파일을 망가뜨려서 `vqa.submit` 이 막는지 보고, 그다음
제출 CSV를 직접 망가뜨려서 `check_submission.py` 가 막는지 봅니다.

In [ ]:
failures = []

for mode in ("missing-id", "duplicate-id", "nan", "bad-sum"):
    broken_pred = FAKE_DIR / ("broken_%s.csv" % mode)
    script("make_fake_predictions.py",
           "--ids", TEST_CSV,
           "--out", str(broken_pred),
           "--mode", "random",
           "--run-id", FAKE_RUN_ID + "_broken",
           "--break", mode, "--break-count", "3",
           "--seed", SEED,
           quiet=True)
    res = vqa("submit", "--pred", str(broken_pred), "--sample", SAMPLE,
              "--out", "artifacts/submission/broken_%s.csv" % mode,
              check=False, quiet=True)
    caught = res["rc"] != 0
    failures.append((mode, caught))
    print("%-14s -> %s" % (mode, "막았습니다" if caught else "통과시켰습니다"))

In [ ]:
BROKEN_SUB = "artifacts/submission/fake_broken.csv"
rows = read_csv_rows(FAKE_SUB)
rows = rows[:-3]                     # 행을 몇 개 지웁니다.
if rows:
    rows[0]["answer"] = "e"          # 허용되지 않는 값을 넣습니다.
    rows.append(dict(rows[0]))       # 중복 ID를 만듭니다.
with open(BROKEN_SUB, "w", newline="", encoding="utf-8") as fh:
    w = csv.DictWriter(fh, fieldnames=["id", "answer"])
    w.writeheader()
    for r in rows:
        w.writerow({"id": r["id"], "answer": r["answer"]})

res = script("check_submission.py", "--submission", BROKEN_SUB,
             "--sample", SAMPLE, check=False)
failures.append(("제출 CSV 손상", res["rc"] != 0))

print()
missed = [m for m, caught in failures if not caught]
if missed:
    print("문제: 다음 결함을 잡지 못했습니다. 검사 항목을 보완해야 합니다.")
    for m in missed:
        print("  -", m)
else:
    print("PASS: 준비한 결함을 모두 잡아냈습니다.")

## 7. 실제 예측 파일로 전환

위 경로가 모두 통과했으면, 파라미터 셀의 `REAL_*` 값을 실제 파일 경로로 바꾸고 아래 셀을
실행합니다. 셀 구조는 위와 같으며 입력만 달라집니다.

아직 준비되지 않은 출처는 빈 문자열로 두면 그 출처를 뺀 채로 비교합니다. 예를 들어 27B 미세조정이
끝나기 전이라면 `REAL_VAL_FT27` 만 비워 두면 됩니다.

In [ ]:
def source_args(prefix, mapping):
    args = []
    for name, path in mapping.items():
        if path and Path(path).exists():
            args += ["--%s-%s" % (prefix, name), path]
        elif path:
            print("아직 없는 파일이라 제외합니다:", path)
    return args


val_args = source_args("val", {"base27": REAL_VAL_BASE27,
                               "ft27": REAL_VAL_FT27,
                               "ft9": REAL_VAL_FT9})
test_args = source_args("test", {"base27": REAL_TEST_BASE27,
                                 "ft27": REAL_TEST_FT27,
                                 "ft9": REAL_TEST_FT9})

print("validation 출처 %d개, test 출처 %d개" % (len(val_args) // 2, len(test_args) // 2))

In [ ]:
if not val_args:
    print("아직 실제 validation 확률 파일이 하나도 없습니다. 준비되면 다시 실행하십시오.")
else:
    for label, pred in (("base27", REAL_VAL_BASE27),
                        ("ft27", REAL_VAL_FT27),
                        ("ft9", REAL_VAL_FT9)):
        if not (pred and Path(pred).exists()):
            continue
        print("=" * 60)
        print("단독 성능:", label)
        script("evaluate_run.py",
               "--pred", pred, "--gold", GOLD, "--val-ids", VAL_IDS_FILE,
               "--json-out", "artifacts/eval/real_%s.json" % label,
               "--note", "단독 %s" % label,
               check=False)

In [ ]:
if not (val_args and test_args):
    print("결합에는 validation과 test 확률이 모두 필요합니다.")
else:
    REAL_SUB = "artifacts/submission/final_candidate.csv"
    script("experiments/eval_tools/combine.py", *val_args, *test_args,
           "--gold", GOLD,
           "--val-ids", VAL_IDS_FILE,
           "--out", "artifacts/eval/real_test_combined.csv",
           "--val-out", "artifacts/eval/real_val_combined.csv",
           "--report-out", "artifacts/eval/real_combine_report.json",
           "--tie-epsilon", str(TIE_EPSILON))

    vqa("submit", "--pred", "artifacts/eval/real_test_combined.csv",
        "--sample", SAMPLE, "--out", REAL_SUB,
        "--margins-out", "artifacts/eval/real_margins.csv")
    script("check_submission.py", "--submission", REAL_SUB, "--sample", SAMPLE)

    print()
    print("===== 최종 제출 후보 =====")
    print("파일   : %s" % REAL_SUB)
    print("sha256 : %s" % sha256_of(REAL_SUB))
    print("결합 보고: artifacts/eval/real_combine_report.json")
    print("=========================")

## 8. 실험 기록

`evaluate_run.py` 와 `experiments/eval_tools/combine.py` 는 실행할 때마다 `artifacts/experiments.jsonl` 에 한 줄씩
덧붙입니다. 여러 사람이 동시에 기록해도 기존 내용을 덮어쓰지 않습니다.

아래 셀이 그 기록을 표로 정리합니다. 마지막 날 최종본을 고를 때 이 표가 근거가 됩니다.
Kaggle 점수처럼 사람만 알 수 있는 값은 그다음 셀로 덧붙입니다.

In [ ]:
LOG_PATH = Path(EXPERIMENTS_LOG)
entries = []
if LOG_PATH.exists():
    for line in LOG_PATH.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line:
            continue
        try:
            entries.append(json.loads(line))
        except ValueError:
            continue

if not entries:
    print("아직 기록된 실험이 없습니다.")
else:
    cols = ["logged_at", "run_id", "accuracy", "n", "seconds_per_item", "note"]
    widths = {c: max(len(c), max(len(str(e.get(c, ""))[:40]) for e in entries)) for c in cols}
    print(" | ".join(c.ljust(widths[c]) for c in cols))
    print("-+-".join("-" * widths[c] for c in cols))
    for e in entries[-40:]:
        print(" | ".join(str(e.get(c, ""))[:40].ljust(widths[c]) for c in cols))
    print()
    print("전체 %d건 가운데 최근 %d건을 보여 주었습니다." % (len(entries), min(40, len(entries))))

In [ ]:
# Kaggle 점수처럼 자동으로 알 수 없는 값은 여기에 적어서 덧붙입니다.
MANUAL_NOTE = {
    "run_id": "",
    "submission": "",
    "sha256": "",
    "kaggle_public": "",
    "selected_as_final": False,
    "note": "",
}

if MANUAL_NOTE["run_id"]:
    MANUAL_NOTE["logged_at"] = time.strftime("%Y-%m-%dT%H:%M:%S")
    MANUAL_NOTE["source"] = "manual"
    LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
    with LOG_PATH.open("a", encoding="utf-8") as fh:
        fh.write(json.dumps(MANUAL_NOTE, ensure_ascii=False) + "\n")
    print("기록을 추가했습니다:", LOG_PATH)
else:
    print("run_id가 비어 있어 추가하지 않았습니다. 기록할 내용이 있으면 MANUAL_NOTE를 채우십시오.")